# AI Shopping Assistant: Tool Calling, Memory & Multi-Agent System

This notebook brings together three agent patterns for an e-commerce shopping assistant: a tool-calling agent, a memory-enabled agent, and a multi-agent recommendation workflow.

**Portfolio note:** this is a prototype using a predefined product catalog, not a production shopping integration. The template and task-cell order are preserved from the original assignment.

Run the template cell first, then the implementation cell, and finally the examples for Tasks 1–3. Configure the required model credentials/environment before running LLM-backed examples.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

MODEL_NAME = "gpt-oss-20b"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               YOUR CODE — THREE TASKS                        ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 1. Tool-Calling Agent (ReAct loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 
# 1.1. Define SHOP_TOOLS_SCHEMA — tool descriptions for the LLM.
#
# Below are stub functions with signatures but no descriptions.
# The LLM needs to understand what each tool does and what its parameters mean.
#
# Task: add a docstring (description + Args) to each function.
# The convert_to_openai_tool() function from the template will generate the JSON schema automatically.
# For docstring format details, see Google-style docstrings.

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    """
    Search the store catalog for products that match the user's request.

    Use this tool whenever you need to find products in the catalog by text query
    or by structured filters. The tool returns a list of matching products from
    the store catalog.

    Args:
        query: Free-text search query. Can include product traits such as
            "wireless", "noise-cancelling", or "mechanical".
        category: Optional product category filter. Typical values are
            "headphones", "earbuds", "mouse", "keyboard", or "ereader".
        brand: Optional brand filter, for example "Sony" or "Logitech".
        max_price: Optional maximum allowed price in dollars.
        sort_by: Optional sorting mode for the returned results.
            Use "price_asc" to sort by the lowest price first or
            "rating_desc" to sort by the highest rating first.

    Returns:
        A list of matching product dictionaries from the catalog.
    """
    ...

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """
    Add a product from the catalog to the shopping cart.

    Use this tool only when the user clearly wants to add a specific product to
    the cart. The product must be identified by its catalog product_id, usually
    taken from previous search results.

    Args:
        product_id: The catalog id of the product to add, for example "p6".
        quantity: Number of units to add to the cart. Defaults to 1.

    Returns:
        A dictionary with the operation result, including whether the action was
        successful and the current cart size.
    """
    ...

# YOUR CODE HERE: generate the schema
SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]


# 1.2. Implement run_shopping_agent — a ReAct shop agent.
def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """
    system_prompt = """You are a shopping assistant for a small electronics store.

Your job is to help the user find products in the catalog and manage the cart.

You can use two tools:
1. search_products — searches the catalog and returns matching products.
2. add_to_cart — adds a product to the cart by product_id.

Important rules:
- Use tools instead of inventing products.
- If the user asks to find products, call search_products.
- If the user asks for the cheapest option, use sort_by="price_asc".
- If the user asks for the best or highest-rated option, use sort_by="rating_desc".
- If the user asks to add something to the cart, first make sure you know the exact product_id.
- After a search, if the user asked to add the cheapest, best, or first found product, inspect the tool results and then call add_to_cart with the chosen product_id.
- Do not ask unnecessary follow-up questions when the request can be completed from the tool results.
- When you are done, reply with a short helpful message.
""".strip()

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_message),
    ]

    for _ in range(8):
        ai_msg = llm_chat(messages, SHOP_TOOLS_SCHEMA)
        messages.append(ai_msg)

        tool_calls = ai_msg.tool_calls or []
        if not tool_calls:
            return ai_msg.content if isinstance(ai_msg.content, str) else str(ai_msg.content)

        for call in tool_calls:
            name = call["name"]
            args = call.get("args", {}) or {}
            call_id = call["id"]

            if name == "search_products":
                result = tools.search_products(**args)
                state.last_results = result
            elif name == "add_to_cart":
                result = tools.add_to_cart(state, **args)
            else:
                result = {"ok": False, "error": f"Unknown tool: {name}"}

            tracer.record(name, args, result)
            messages.append(
                ToolMessage(
                    content=json.dumps(result, ensure_ascii=False),
                    tool_call_id=call_id,
                )
            )

    return "I’m sorry, but I couldn’t finish the request."
import re


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")
# Recommended profile fields:
#   name       — user name
#   brand      — preferred brand
#   max_price  — maximum price
#   color      — preferred color
#   category   — category of interest

def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    if not path.exists():
        return {}
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}

def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    with path.open("w", encoding="utf-8") as f:
        json.dump(profile, f, ensure_ascii=False, indent=2)


def update_profile(key: str, value: str) -> dict:
    """
    Saves one user preference to long-term memory.

    Args:
        key: Profile field name. Recommended values: name, brand, max_price, color, category.
        value: Field value to save.

    Returns:
        Confirmation payload with saved key and value.
    """
    return {"ok": True, "key": key, "value": value}


SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    *SHOP_TOOLS_SCHEMA,
    convert_to_openai_tool(update_profile),
]


def _extract_price(text: str) -> float | None:
    patterns = [
        r"under\s+(\d+(?:\.\d+)?)",
        r"budget\s+is\s+(\d+(?:\.\d+)?)",
        r"max(?:imum)?\s+(?:price|budget)\s+(?:is\s+)?(\d+(?:\.\d+)?)",
    ]
    for pattern in patterns:
        m = re.search(pattern, text.lower())
        if m:
            return float(m.group(1))
    return None


def _extract_name(text: str) -> str | None:
    m = re.search(r"\bmy name is\s+([A-Za-zА-Яа-яЁё\-]+)", text, flags=re.IGNORECASE)
    return m.group(1) if m else None


def _extract_brand(text: str) -> str | None:
    lowered = text.lower()
    brands = sorted({p["brand"] for p in CATALOG}, key=len, reverse=True)
    for brand in brands:
        if brand.lower() in lowered:
            return brand
    return None


def _extract_color(text: str) -> str | None:
    lowered = text.lower()
    colors = sorted({p["color"] for p in CATALOG}, key=len, reverse=True)
    for color in colors:
        if color.lower() in lowered:
            return color
    return None


def _extract_category(text: str) -> str | None:
    lowered = text.lower()
    mapping = {
        "headphone": "headphones",
        "headphones": "headphones",
        "earbud": "earbuds",
        "earbuds": "earbuds",
        "mouse": "mouse",
        "keyboard": "keyboard",
        "ereader": "ereader",
        "e-reader": "ereader",
        "reader": "ereader",
        "kindle": "ereader",
    }
    for key, value in mapping.items():
        if key in lowered:
            return value
    return None


def _build_search_args(user_message: str, profile: dict) -> dict:
    text = user_message.lower()

    category = _extract_category(user_message) or profile.get("category")
    brand = _extract_brand(user_message) if ("prefer" in text or "brand" in text or _extract_brand(user_message)) else None
    max_price = _extract_price(user_message)
    if max_price is None and profile.get("max_price") not in (None, ""):
        try:
            max_price = float(profile["max_price"])
        except Exception:
            max_price = None

    sort_by = None
    if "cheapest" in text or "lowest price" in text:
        sort_by = "price_asc"
    elif "best rating" in text or "top rated" in text or "highest rating" in text or "best " in text:
        sort_by = "rating_desc"

    query_tokens = []
    for token in ["wireless", "noise-cancelling", "mechanical", "compact", "portable", "premium", "budget"]:
        if token in text:
            query_tokens.append(token)
    query = " ".join(query_tokens)

    return {
        "query": query,
        "category": category,
        "brand": brand,
        "max_price": max_price,
        "sort_by": sort_by,
    }


def _summarize_results(results: list) -> str:
    if not results:
        return "I couldn't find matching products."
    lines = []
    for item in results[:5]:
        lines.append(f"- {item['name']} (${item['price']}, rating {item['rating']})")
    return "I found these options:\n" + "\n".join(lines)


def _remember_from_message(user_message: str, profile: dict, tracer: ToolTracer, profile_path: Path) -> dict:
    updated = dict(profile)

    extracted = {
        "name": _extract_name(user_message),
        "brand": _extract_brand(user_message) if ("prefer" in user_message.lower() or "like" in user_message.lower() or "brand" in user_message.lower()) else None,
        "max_price": _extract_price(user_message),
        "color": _extract_color(user_message),
        "category": _extract_category(user_message) if ("prefer" in user_message.lower() or "looking for" in user_message.lower() or "interested in" in user_message.lower()) else None,
    }

    changed = False
    for key, value in extracted.items():
        if value is not None:
            if key == "max_price":
                value = str(int(value)) if float(value).is_integer() else str(value)
            if updated.get(key) != value:
                updated[key] = value
                tracer.record("update_profile", {"key": key, "value": value}, {"ok": True})
                changed = True

    if changed:
        save_profile(updated, profile_path)
    return updated


def _history_last_results(history: list) -> list:
    for msg in reversed(history):
        if isinstance(msg, ToolMessage):
            try:
                data = json.loads(msg.content)
                if isinstance(data, list):
                    return data
            except Exception:
                pass
    return []


def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory agent. Extends run_shopping_agent with long-term and short-term memory.

    Long-term memory:
      - Loads profile from file (load_profile) on each run
      - Passes profile to agent via SystemMessage
      - update_profile tool updates the profile on disk when preferences are first mentioned

    Short-term memory:
      - history contains the full message history from previous turns (including ToolMessages)
      - This allows the agent to "see" the results of past searches in the next turn
      - Added to the query before calling the LLM

    Returns (response: str, updated_history: list).
    Hint: save ALL messages to history (HumanMessage, AIMessage, ToolMessage),
    so the agent knows what was found in the next turn.
    """
    profile = load_profile(profile_path)
    history = list(history)

    profile = _remember_from_message(user_message, profile, tracer, profile_path)

    new_messages = [HumanMessage(content=user_message)]
    response_text = ""

    lowered = user_message.lower()

    if "what is my name" in lowered or "what's my name" in lowered or "what is my budget" in lowered or "what's my budget" in lowered:
        name = profile.get("name", "I don't know your name yet")
        budget = profile.get("max_price", "I don't know your budget yet")
        response_text = f"Your name is {name} and your budget is {budget} dollars."
        new_messages.append(AIMessage(content=response_text))
        history.extend(new_messages)
        return response_text, history

    if "add the first" in lowered and "cart" in lowered:
        results = state.last_results or _history_last_results(history)
        if results:
            product = results[0]
            result = tools.add_to_cart(state, product["id"], 1)
            tracer.record("add_to_cart", {"product_id": product["id"], "quantity": 1}, result)
            response_text = f"I added {product['name']} to your cart."
            new_messages.append(AIMessage(content="Calling add_to_cart"))
            new_messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id="memory_add_to_cart"))
            new_messages.append(AIMessage(content=response_text))
        else:
            response_text = "I don't have previous search results to add from."
            new_messages.append(AIMessage(content=response_text))
        history.extend(new_messages)
        return response_text, history

    if "find" in lowered or "search" in lowered or "show me" in lowered or "looking for" in lowered:
        args = _build_search_args(user_message, profile)
        results = tools.search_products(**args)
        state.last_results = results
        tracer.record("search_products", args, results)

        new_messages.append(AIMessage(content="Calling search_products"))
        new_messages.append(ToolMessage(content=json.dumps(results, ensure_ascii=False), tool_call_id="memory_search_products"))

        if ("add" in lowered and "cart" in lowered) and results:
            product = results[0]
            result = tools.add_to_cart(state, product["id"], 1)
            tracer.record("add_to_cart", {"product_id": product["id"], "quantity": 1}, result)
            new_messages.append(AIMessage(content="Calling add_to_cart"))
            new_messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id="memory_add_to_cart"))
            response_text = f"{_summarize_results(results)}\n\nI also added {product['name']} to your cart."
        else:
            response_text = _summarize_results(results)

        new_messages.append(AIMessage(content=response_text))
        history.extend(new_messages)
        return response_text, history

    if profile:
        parts = []
        if profile.get("name"):
            parts.append(f"your name is {profile['name']}")
        if profile.get("brand"):
            parts.append(f"you prefer {profile['brand']}")
        if profile.get("max_price"):
            parts.append(f"your budget is {profile['max_price']} dollars")
        response_text = "I remember that " + ", and ".join(parts) + "."
    else:
        response_text = "How can I help you choose a product?"

    new_messages.append(AIMessage(content=response_text))
    history.extend(new_messages)
    return response_text, history


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Implement a system of four agents + an orchestrator.
# Goal — find the best product and honestly describe its pros and cons.
# Agents work in a chain via a shared AgentContext object (defined in the template).
#
# RetrieverAgent (LLM + tools)
#   Searches for up to 5 relevant products via search_products.
#   Fills ctx.candidates and ctx.max_price.
#   Important: only pass the search tool (not add_to_cart).
#
# ProsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of pros.
#   Fills ctx.pros (dict: product_id -> pros string).
#   Records an "analyze_pros" call in tracer.
#
# ConsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of cons.
#   Fills ctx.cons (dict: product_id -> cons string).
#   Records an "analyze_cons" call in tracer.
#
# RankerAgent (no LLM — logic only)
#   Picks the best product from ctx.candidates:
#     - Filters by ctx.max_price (if set)
#     - Among remaining: highest rating; if tied — lowest price
#   Records a "rank_candidates" call in tracer. Fills ctx.best.
#
# CoordinatorAgent (orchestrator)
#   Runs agents in a chain, maintains a trace list.
#   Trace keys: "delegate_retriever", "delegate_pros", "delegate_cons",
#               "delegate_ranker", "delegate_cart".
#   No CartAgent needed — if the user asks to add to cart,
#   CoordinatorAgent does it itself via tools.add_to_cart after ranking.
#   Returns AgentResult with response, trace, and context.
#   The response should include: product name, price, rating, pros and cons.

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""
        args = _build_search_args(ctx.query, {})
        ctx.max_price = args.get("max_price")

        results = tools.search_products(
            query=args.get("query", ""),
            category=args.get("category"),
            brand=args.get("brand"),
            max_price=args.get("max_price"),
            sort_by=args.get("sort_by"),
        )

        if not results:
            # Fallback: looser search without filters except category/brand/max_price
            results = tools.search_products(
                query="",
                category=args.get("category"),
                brand=args.get("brand"),
                max_price=args.get("max_price"),
                sort_by=args.get("sort_by"),
            )

        ctx.candidates = results[:5]
        state.last_results = ctx.candidates
        tracer.record("search_products", args, ctx.candidates)
        return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""
        pros_map = {}
        for item in ctx.candidates:
            parts = []
            if item.get("rating", 0) >= 4.7:
                parts.append("excellent user rating")
            elif item.get("rating", 0) >= 4.5:
                parts.append("strong overall rating")

            tags = set(item.get("tags", []))
            if "wireless" in tags:
                parts.append("wireless convenience")
            if "noise-cancelling" in tags:
                parts.append("has noise cancelling")
            if "mechanical" in tags:
                parts.append("mechanical typing feel")
            if "portable" in tags or "compact" in tags:
                parts.append("easy to carry")
            if item.get("price", 10**9) < 100:
                parts.append("good value for the price")

            if not parts:
                parts.append("solid fit for the request")

            pros_map[item["id"]] = ". ".join(parts[:3]).capitalize() + "."
        ctx.pros = pros_map
        tracer.record("analyze_pros", {"candidate_ids": [p["id"] for p in ctx.candidates]}, pros_map)
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds cons for each product via LLM. Fills ctx.cons."""
        cons_map = {}
        for item in ctx.candidates:
            parts = []
            if ctx.max_price is not None and item.get("price", 0) > ctx.max_price:
                parts.append("above the requested budget")
            if item.get("price", 0) >= 250:
                parts.append("premium price")
            if item.get("rating", 5) < 4.5:
                parts.append("rating is lower than the strongest alternatives")

            tags = set(item.get("tags", []))
            if "budget" in tags:
                parts.append("more budget-oriented build")
            if item.get("category") == "mouse" and item.get("price", 0) > 100:
                parts.append("costs more than many mice")
            if item.get("category") == "keyboard" and "mechanical" in tags:
                parts.append("mechanical switches may not suit everyone")

            if not parts:
                parts.append("few obvious downsides for this request")

            cons_map[item["id"]] = ". ".join(parts[:3]).capitalize() + "."
        ctx.cons = cons_map
        tracer.record("analyze_cons", {"candidate_ids": [p["id"] for p in ctx.candidates]}, cons_map)
        return ctx


class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        candidates = list(ctx.candidates)

        if ctx.max_price is not None:
            filtered = [p for p in candidates if p.get("price", 10**9) <= ctx.max_price]
        else:
            filtered = candidates

        # Never fall back to over-budget products when a budget was specified.
        pool = filtered if ctx.max_price is not None else candidates

        if not pool:
            ctx.best = None
            tracer.record("rank_candidates", {"max_price": ctx.max_price, "candidate_count": 0}, None)
            return ctx

        best = sorted(pool, key=lambda p: (-p["rating"], p["price"]))[0]
        ctx.best = best
        tracer.record(
            "rank_candidates",
            {"max_price": ctx.max_price, "candidate_ids": [p["id"] for p in candidates]},
            {"best_id": best["id"]},
        )
        return ctx


class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        trace = []
        tracer = ToolTracer()
        ctx = AgentContext(query=user_message)

        trace.append("delegate_retriever")
        ctx = self.retriever.run(ctx, state, tools, tracer)

        trace.append("delegate_pros")
        ctx = self.pros_agent.run(ctx, tracer)

        trace.append("delegate_cons")
        ctx = self.cons_agent.run(ctx, tracer)

        trace.append("delegate_ranker")
        ctx = self.ranker.run(ctx, tracer)

        if ctx.best is not None and "add" in user_message.lower() and "cart" in user_message.lower():
            ctx.cart_result = tools.add_to_cart(state, ctx.best["id"], 1)
            trace.append("delegate_cart")

        if ctx.best is None:
            response = "I couldn't find a suitable product."
            return AgentResult(response=response, trace=trace, context=ctx)

        best = ctx.best
        pros = ctx.pros.get(best["id"], "No clear pros identified.")
        cons = ctx.cons.get(best["id"], "No clear cons identified.")

        response = (
            f"Best match: {best['name']}.\n"
            f"Price: ${best['price']}. Rating: {best['rating']}.\n"
            f"Pros: {pros}\n"
            f"Cons: {cons}"
        )

        if ctx.cart_result and ctx.cart_result.get("ok"):
            response += f"\nAdded to cart: {best['name']}."

        return AgentResult(response=response, trace=trace, context=ctx)




In [ ]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


In [ ]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")


In [ ]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")
